# Reproducing Gable et al. (2026), Figure S4 and Table 5, from DANDI:000010

**Paper.** Gable J, Newman ZL, Young S, Scheib J, Bliese S, Miller N, Kerlin A (2026). Tuft dendrite
spikes are accompanied by selective input from distinct functional networks. *J Neurosci* 46(38)
e0974262026, [doi:10.1523/jneurosci.0974-26.2026](https://doi.org/10.1523/jneurosci.0974-26.2026).
The claims file (`claims/gable-2026.yaml`) was written from the version of record (PMC13612898)
with its supplement, and from the paper's code, github.com/kerlin-lab/Gable_2026 at commit 688e662.

**Dataset.** [DANDI:000010](https://dandiarchive.org/dandiset/000010/0.220126.1905), version
0.220126.1905, "A motor cortex circuit for motor planning and movement" (Li, Chen, Guo, Gerfen and
Svoboda 2015). The paper reuses its silicon-probe recordings of ALM for Figure S4 and Table 5 only.
It cites the dataset without a version. 0.220126.1905 is the only published version; the draft
holds the same 158 assets (same paths, asset ids and sizes, checked below) and differs only in
dandiset metadata, so the published version is pinned.

**What can be tested.** The four claims on the silicon-probe recordings (figs4-*). Every other claim
rests on the paper's own two-photon imaging, which is not in this dataset nor in any public archive
the paper or its code names. The claims file marks them `data: dataset`, and they are blocked as
`data_missing` (a reader's answer in `questions/gable-2026.yaml` changed them from `data: elsewhere`).

| Claim | Tested here |
|---|---|
| figs4-session-selection | yes |
| figs4-lowest-couplers-correlated | yes |
| figs4-high-couplers-correlated | yes |
| figs4-middle-couplers-not-correlated | yes (run; blocked as `methods_underspecified`, see its section) |
| 30 imaging claims (fig1 to fig5, figs1 to figs3, table1) | no, blocked (`data_missing`): the imaging data are not public |

**Method.** The claims file's E1 to E8 (selection, binning, trial-specific activity, population
coupling, pairs, bootstrap test) are implemented in `alm_coupling.py`, whose docstring says where
each parameter comes from. The open details O1 to O3 are run under every reading:

- O1 (a) all high-quality units of a selected session, (b) only those above 0.5 Hz;
- O2 (a) rate over the time each unit was observed, (b) rate within the analysis windows;
- O3 (a) only the four trial types CL, CR, IL, IR, (b) all trials with sample and GO times.

Each combination is written as three letters, `aaa` to `bbb`. A fourth detail, which the claims
file does not name, is how to treat units stored twice in a file (see the data check): counted once
(`baa`) or every stored row kept (`baa-r`). All sixteen combinations are run through every claim.

The paper's code at commit 688e662 is read for its printed outputs (FigureS4.ipynb: the neurons per
session and the pair counts per bin), which are compared with each reading; it is an input of the
attempt, fetched from GitHub at the pinned commit.

In [ ]:
import re
import resource
import time
from collections import Counter
import warnings
import zlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

import alm_coupling as ac

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

PAPER = "gable-2026"
started = time.time()
assets = ac.ecephys_assets()
sessions = ac.load_all(assets)
url_of = dict(assets)
print(f"loaded {len(sessions)} ecephys files of dandiset {ac.DANDISET_ID} version {ac.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
api = "https://api.dandiarchive.org/api/dandisets/000010/versions/{}/assets/?page_size=500"
listing = {v: {(a["path"], a["asset_id"], a["size"]) for a in requests.get(api.format(v)).json()["results"]}
           for v in [ac.DANDISET_VERSION, "draft"]}
print("assets, published:", len(listing[ac.DANDISET_VERSION]), " draft:", len(listing["draft"]),
      " differing (path, asset id, size):", len(listing[ac.DANDISET_VERSION] ^ listing["draft"]))

info = pd.DataFrame([{
    "subject": s["subject"], "session": s["session"], "genotype": s["genotype"], "strains": s["strains"],
    "group": ac.strain_group(s["strains"]), "sex": s["sex"],
    "trials": len(s["trials"]), "task": "/".join(sorted(s["trials"].task.unique())),
    "units": len(s["units"]), "unique_units": len(ac.unique_units(s)),
    "quality": "/".join(sorted(s["units"].quality.unique())),
    "photostim_events": int(s["trials"].photostim_start_time.notna().sum()),
    "events_nan_time": int(sum(s["unmatched_events"].values())),
} for s in sessions])
print("\nFiles:", len(info), " subjects:", info.subject.nunique())
print("\nMice and sessions per genotype (subject description):")
print(info.groupby(["group", "genotype", "strains"], dropna=False).agg(
    mice=("subject", "nunique"), sessions=("subject", "size"), units=("units", "sum")).to_string())
print("\nQuality labels over all units:", info.quality.unique().tolist())
print("Task label of the trials:", info.task.unique().tolist())
print("Cell types:", pd.Series(np.concatenate([s["units"].cell_type for s in sessions])).value_counts().to_dict())
print("Units per file: min", info.units.min(), "median", info.units.median(), "max", info.units.max())
print("\n", info.drop(columns=["strains"]).to_string(index=False))

The subject records give five genotypes. KJ18 is the Sim1-Cre line and PL56 the Tlx3-Cre line
named by the paper; ten mice carry KJ18 (one of them crossed with Ai32) and six PL56 (two crossed
with Ai32); three Rbp4-Cre (KL100) mice are not among the paper's genotypes and are left out. Every
unit is labelled "good", so the paper's "high-quality units, as labeled in the original dataset"
keeps every unit. Every trial is labelled `audio delay`, while the dataset's experiment description
("object location discrimination task") and the paper (tactile sample cue) describe a tactile
task: the task label disagrees with the description.

In [ ]:
dup_rows, dup_identical, files_with_dups = 0, 0, 0
for s in sessions:
    ids = s["units"].id.to_numpy()
    vals, counts = np.unique(ids, return_counts=True)
    if (counts > 1).any():
        files_with_dups += 1
    for v in vals[counts > 1]:
        w = np.where(ids == v)[0]
        dup_rows += len(w) - 1
        dup_identical += all(np.array_equal(s["spikes"][w[0]], s["spikes"][j]) for j in w[1:])
print(f"files with a repeated unit id: {files_with_dups} of {len(sessions)}; extra rows: {dup_rows}; "
      f"repeated ids whose rows hold identical spike trains: {dup_identical}")
print("units in all files:", int(info.units.sum()), " counted once:", int(info.unique_units.sum()))
example = info[info.units > info.unique_units].iloc[0]
s = next(x for x in sessions if x["subject"] == example.subject and x["session"] == example.session)
print("example", s["path"], "unit ids:", s["units"].id.tolist())

# identical spike trains across files (a recording deposited twice)
keys = {}
for s in sessions:
    for j in ac.unique_units(s):
        keys.setdefault((len(s["spikes"][j]), s["spikes"][j][:20].tobytes()), set()).add(s["path"])
print("spike trains found in more than one file:", sum(len(v) > 1 for v in keys.values()))
starts = {}
for s in sessions:
    starts.setdefault(np.round(s["trials"].start.to_numpy(), 3).tobytes(), []).append(s["path"])
print("files sharing an identical list of trial start times:", [v for v in starts.values() if len(v) > 1])

# spike times against the observation intervals
rows = []
for s in sessions:
    ob0 = s["obs"][0]
    tr = s["trials"]
    has = tr["sample"].notna().to_numpy()
    out = 0
    for sp, ob in zip(s["spikes"], s["obs"]):
        i = np.searchsorted(ob[:, 0], sp, side="right") - 1
        out += int(np.sum(~((i >= 0) & (sp <= ob[np.clip(i, 0, None), 1]))))
    rows.append({"path": s["path"], "units_share_intervals": all(np.array_equal(o, ob0) for o in s["obs"]),
                 "intervals": len(ob0), "trials_with_sample": int(has.sum()),
                 "trials_with_interval": int(np.isfinite(ac.observed_until(s)).sum()),
                 "sample_but_no_interval": int((has & ~np.isfinite(ac.observed_until(s))).sum()),
                 "interval_lengths": "/".join(map(str, np.unique(np.round(ob0[:, 1] - ob0[:, 0], 3)))),
                 "stop_minus_interval_end_max": float(np.nanmax(np.abs(tr.stop.to_numpy() - ac.observed_until(s)))),
                 "spikes_outside": out, "unsorted_units": s["unsorted_units"]})
obs_check = pd.DataFrame(rows)
print("files whose units share one set of observation intervals:", int(obs_check.units_share_intervals.sum()), "of", len(obs_check))
print("intervals:", int(obs_check.intervals.sum()), " intervals starting at a trial start:", int(obs_check.trials_with_interval.sum()),
      " trials with a sample time:", int(obs_check.trials_with_sample.sum()),
      " of these without an interval:", int(obs_check.sample_but_no_interval.sum()),
      "in", int((obs_check.sample_but_no_interval > 0).sum()), "files")
print("interval lengths (s):", obs_check.interval_lengths.unique().tolist())
print("largest difference between a trial's stop time and its interval's end (s):",
      round(obs_check.stop_minus_interval_end_max.max(), 3))
print("spikes outside their unit's observation intervals, by file:")
print(obs_check[obs_check.spikes_outside > 0][["path", "spikes_outside"]].to_string(index=False))
print("units whose spike times are stored out of order (sorted on loading), by file:")
print(obs_check[obs_check.unsorted_units > 0][["path", "unsorted_units"]].to_string(index=False))

In 50 of the 99 files some unit ids appear twice; in every case the two rows hold the same spike
train (for example ids 3, 4, 4, 5, 5), so the file stores one unit twice. Whether another unit
was lost in its place cannot be told from the files. Units are counted once here. No spike train or list of trial
starts is repeated across files. Every unit of a file shares one set of observation intervals,
one per trial with a sample time, starting at the trial start and lasting 5.5 s; the trials'
stop times differ from the interval ends. Spikes are stored only within these intervals (apart
from the files listed), so a unit's "session-wide" rate is its rate over those intervals. A few
units store spike times out of order; they are sorted on loading.

In [ ]:
allt = pd.concat([s["trials"].assign(path=s["path"]) for s in sessions], ignore_index=True)
print("trials:", len(allt))
print("\noutcome by early lick:")
print(pd.crosstab(allt.early_lick, allt.outcome).to_string())
print("\ntrials without a sample time, by outcome:")
print(allt[allt["sample"].isna()].outcome.value_counts().to_string())
d_sd = (allt.delay - allt["sample"]).dropna()
d_dg = (allt.go - allt.delay).dropna()
print(f"\nsample to delay onset (s): median {d_sd.median():.4f}, within 10 ms of 1.3 s: {np.mean(abs(d_sd - 1.3) < 0.01):.3f}")
print(f"delay to GO (s): median {d_dg.median():.4f}, within 10 ms of 1.3 s: {np.mean(abs(d_dg - 1.3) < 0.01):.3f}")
print(f"trial length (stop - start, s): median {(allt.stop - allt.start).median():.2f}, "
      f"range {(allt.stop - allt.start).min():.2f} to {(allt.stop - allt.start).max():.2f}")
print(f"sample onset after trial start (s): median {(allt['sample'] - allt.start).median():.3f}")
ph = allt.dropna(subset=["photostim_start_time"])
print(f"\nphotostimulation onset relative to sample onset (s): median {(ph.photostim_start_time - ph['sample']).median():.3f}, "
      f"range {(ph.photostim_start_time - ph['sample']).min():.3f} to {(ph.photostim_start_time - ph['sample']).max():.3f}")
print("events with a NaN timestamp, by event:",
      pd.DataFrame([s["unmatched_events"] for s in sessions]).sum().astype(int).to_dict())

The sample and delay epochs are each 1.3 s in most trials (Li et al. 2015), so the window from
sample onset to 2 s after GO is 4.6 s; an early lick restarts the epoch and lengthens the window.
Some events in the BehavioralEvents series have a NaN timestamp; they cannot be placed in a trial.
Whether a trial was photostimulated is read from the photostimulation onsets; it is checked below
against the stored laser trace for the selected sessions.

### Clock check

The spikes and the trial events must share a clock. ALM activity predicts the instructed lick
direction after the sample cue and most strongly after GO (Li et al. 2015), and should not predict
it before the sample cue. The instructed side of correct trials is decoded (nearest class mean,
five interleaved folds, balanced accuracy) from each unit's spike count in windows relative to
sample onset, in every selected session of reading `baa-r` (adopted below); as a control each trial
takes the next trial's label.

In [ ]:
main = ("b", "a", "a")   # O1 to O3 of the reading adopted below, under "Population coupling"
MAIN = "baa-r"           # with every stored row kept
W = [(-0.5, 0.0), (0.0, 0.5), (0.8, 1.3), (1.8, 2.6), (2.6, 3.1), (3.1, 3.6), (3.6, 4.6)]
wnames = ["-0.5-0 (pre-sample)", "0-0.5 (sample)", "0.8-1.3", "1.8-2.6 (delay)", "2.6-3.1 (after GO)", "3.1-3.6", "3.6-4.6"]
rows, rows_shift = [], []
for s in sessions:
    if ac.strain_group(s["strains"]) is None:
        continue
    sel = ac.select_session(s, *main, dedupe=False)
    if not sel["passes"]:
        continue
    rows.append(ac.decode_side(s, sel["units"], sel["trial_mask"], "sample", W))
    rows_shift.append(ac.decode_side(s, sel["units"], sel["trial_mask"], "sample", W, shift_labels=True))
clock = pd.DataFrame({"window from sample onset (s)": wnames,
                      "median accuracy": np.median(rows, 0), "min": np.min(rows, 0), "max": np.max(rows, 0),
                      "median, labels shifted one trial": np.median(rows_shift, 0)})
print(f"sessions: {len(rows)}")
print(clock.round(3).to_string(index=False))

The accuracy printed before sample onset and with shifted labels is the chance reference; the
accuracies after sample onset and after GO are compared with it in the cell's output. A rise only
after the cue, and the loss of that rise with shifted labels, is consistent with spikes and trial
events on one clock.

In [ ]:
# photostimulated trials: onsets in BehavioralEvents against the stored laser power trace, in every
# session selected under reading baa (units counted once) or baa-r (every stored row)
rows = []
for s in sessions:
    if ac.strain_group(s["strains"]) is None:
        continue
    sel = ac.select_session(s, *main, dedupe=False)
    if not (sel["passes"] or ac.select_session(s, *main, dedupe=True)["passes"]):
        continue
    mx, names, ogen_meta = ac.laser_max_per_trial(url_of[s["path"]], s["trials"])
    ph = s["trials"].photostim_start_time.notna().to_numpy()
    rows.append({"session": s["path"].split("/")[1][:23], "ai32": "ai32" in s["strains"],
                 "ogen_in_name": "ogen" in s["path"], "optogenetics_metadata": ogen_meta,
                 "laser": "/".join(names) or "none",
                 "onset_trials": int(ph.sum()), "onset_trials_with_power>1": int((mx[ph] > 1).sum()),
                 "other_trials_with_power>1": int((mx[~ph] > 1).sum()),
                 "included_trials_with_power>1": int((mx[sel["trial_mask"]] > 1).sum()),
                 "max_power": float(mx.max())})
laser = pd.DataFrame(rows)
print(laser.to_string(index=False))
print("\nselected sessions of Ai32 mice:", int(laser.ai32.sum()), "; of these with a laser trace or photostimulation onsets:",
      int((laser.ai32 & ((laser.laser != "none") | (laser.onset_trials > 0))).sum()))

In the sessions with a laser trace, every photostimulation onset has laser power and no included
trial has any. The selected sessions of the Ai32 mice (KJ18 or PL56 crossed with ChR2) hold no laser
trace, no photostimulation onsets and no optogenetics metadata, so whether their trials had light
cannot be checked from the files; they are kept, as nothing in them marks a trial as
photostimulated, and the selection is also shown without them.

## figs4-session-selection

> The reanalysis of the Li et al. (2015) ALM silicon-probe recordings includes eight Sim1_KJ18-Cre
> and three Tlx_PL56-Cre mice, 20 sessions, and yields 753 same-coupling-range and 6,586
> different-coupling-range neuron pairs.

Criterion: under one reading of O1 to O3, mice exactly 11 (8 Sim1, 3 Tlx), sessions 20 ± 2, same
753 ± 75, different 6,586 ± 659 (summed over the 7 bins). Mice are counted by the strain of the
subject record; the KJ18 and PL56 mice crossed with Ai32 are counted with their Cre line, and the
count without them is shown too. Each reading is run with units stored twice counted once and with
every stored row. Beside the result are the counts printed by the paper's FigureS4.ipynb, read from
the notebook at commit 688e662.

In [ ]:
READINGS = [(a, b, c, d) for d in (True, False) for a in "ab" for b in "ab" for c in "ab"]
results = {"".join(r[:3]) + ("" if r[3] else "-r"): ac.analyse(sessions, *r[:3], dedupe=r[3]) for r in READINGS}
NAMES = list(results)

code_nb = requests.get("https://raw.githubusercontent.com/kerlin-lab/Gable_2026/"
                       "688e662767ed32b6ea60a2827c3483408edf6deb/Figures/FigureS4.ipynb").json()
code_text = "\n".join("".join(o.get("text", "")) + "".join(o.get("data", {}).get("text/plain", ""))
                      for c in code_nb["cells"] for o in c.get("outputs", []))
code_units = [int(x) for x in re.findall(r"Calculating sess \d+ population coupling:[^\n]*?\| 0/(\d+)", code_text)]
code_same = [int(x) for x in re.findall(r"n_same=(\d+)", code_text)]
code_diff = [int(x) for x in re.findall(r"n_diff=(\d+)", code_text)]
code_md = [float(x) for x in re.findall(r"mean diff = ([-+\d.e]+)", code_text)]
print(f"the code's printed output: {len(code_units)} sessions, {sum(code_units)} neurons, "
      f"{min(code_units)} to {max(code_units)} per session; same-range pairs per bin {code_same} (sum {sum(code_same)}); "
      f"different-range {code_diff} (sum {sum(code_diff)})")
print("the code's neurons per session, sorted:", sorted(code_units))

def counts(res):
    ss = res["sessions"]
    mice = ss.drop_duplicates("subject")
    return {"mice": len(mice), "Sim1": int((mice.group == "Sim1").sum()), "Tlx": int((mice.group == "Tlx").sum()),
            "mice_without_Ai32": int((~mice.ai32).sum()), "sessions": len(ss), "neurons": int(ss.n_units.sum()),
            "units_min": int(ss.n_units.min()), "units_max": int(ss.n_units.max()),
            "sessions_sharing_code_count": sum((Counter(ss.n_units) & Counter(code_units)).values()),
            "same": int(sum(sum(len(v) for v in b) for b in res["same"])),
            "different": int(sum(sum(len(v) for v in b) for b in res["diff"]))}

sel_table = pd.DataFrame([{"reading": n, "units": "counted once" if "-r" not in n else "every stored row",
                           **counts(results[n])} for n in NAMES])
sel_table["match"] = ((sel_table.mice == 11) & (sel_table.Sim1 == 8) & (sel_table.Tlx == 3)
                      & (abs(sel_table.sessions - 20) <= 2) & (abs(sel_table.same - 753) <= 75)
                      & (abs(sel_table.different - 6586) <= 659))
print("paper: 11 mice (8 Sim1, 3 Tlx), 20 sessions, 753 same and 6,586 different pairs")
print(sel_table.to_string(index=False))
print("\nreadings matching all four values:", sel_table[sel_table.match].reading.tolist())
for n in ["baa", "baa-r", "bba", "bba-r"]:
    print(f"{n} neurons per session, sorted:", sorted(results[n]["sessions"].n_units))

In [ ]:
print(f"reading {MAIN}; sessions kept:")
print(results[MAIN]["sessions"][["subject", "session", "group", "ai32", "n_units", "n_trials"]].to_string(index=False))
per_mouse = pd.concat([results[n]["sessions"].assign(reading=n) for n in NAMES])
print("\nsessions per mouse under each reading:")
print(pd.crosstab(per_mouse.subject + " " + per_mouse.group, per_mouse.reading)[NAMES].to_string())
print("\nKJ18 and PL56 mice in the dataset with no selected session under any reading:",
      sorted(set(info[info.group.notna()].subject) - set(per_mouse.subject)))

# candidate sessions per mouse: which filter removes a mouse
rows = []
for s in sessions:
    g = ac.strain_group(s["strains"])
    if g is None:
        continue
    row = {"subject": s["subject"], "group": g, "ai32": "ai32" in s["strains"],
           ">=18 stored rows": len(s["units"]) >= 18, ">=18 units once": len(ac.unique_units(s)) >= 18}
    for d, dn in [(True, "once"), (False, "rows")]:
        for o2 in "ab":
            row[f">=18 above 0.5 Hz, O2 {o2}, {dn}"] = ac.select_session(s, "b", o2, "a", dedupe=d)["passes"]
    rows.append(row)
cand = pd.DataFrame(rows).groupby(["subject", "group", "ai32"]).sum().reset_index()
print("\nsessions per KJ18 and PL56 mouse passing each filter (O3 a; O3 b gives the same sessions):")
print(cand.to_string(index=False))
print("mice with at least one session:", cand.drop(columns=["subject", "group", "ai32"]).gt(0).sum().to_dict())
print("by group:", {c: cand[cand[c] > 0].group.value_counts().to_dict() for c in cand.columns[3:]})

types = pd.DataFrame([{"session": f"{r.subject} {r.session}", **r.types} for r in results["bab-r"]["sessions"].itertuples()]).fillna(0)
print("\ntrials per type in each selected session, reading bab-r (under O3 a only CL, CR, IL, IR are kept):")
print(types.set_index("session").astype(int).to_string())

**Result: not reproduced.** No reading selects the paper's 11 mice. With units counted once every
reading gives 9 mice (7 Sim1, 2 Tlx) and 15 or 16 sessions; with every stored row, 10 mice (7 Sim1,
3 Tlx) and 17 or 18 sessions, against 8 Sim1 and 3 Tlx mice and 20 sessions. The third Tlx mouse
(sub-218453) enters only with every stored row: its session of 18 stored rows holds 16 distinct
units. No reading reaches an eighth Sim1 mouse: nine of the ten KJ18 mice have a session of at least
18 stored units, but only seven have one with 18 units above 0.5 Hz under either rate. Under reading
`baa-r`, adopted below, there are 17 sessions of 18 to 25 neurons (348 neurons, against the code's
20 sessions and 411; 14 of the 17 sessions have a neuron count that appears in the code's list),
575 same-range and 5,686 different-range pairs (paper 753 and 6,586; the code printed 771 and
6,586). Over all sixteen readings the pair counts run from 452 to 893 and from 4,650 to 8,860.

## Population coupling and same against different coupling range

The three remaining claims share one computation (E4 to E8) and differ in the bins they judge. No
reading matched the selection claim, so, as the claims file directs, every reading is run and shown.
A claim's outcome is the one all sixteen readings give. Where they differ, the outcome is taken
under an assumed reading only for a detail with a basis in the paper, its code or the data:

- Units stored twice: every stored row is kept (`-r`). The files give each such unit as two rows of
  the units table, so a pipeline that reads the table as stored keeps both, and the selection
  supports it: only with every row does the third Tlx mouse enter (the paper has three), and the
  sessions (17 against 16) and neurons (348 against 318) come closer to the code's 20 and 411.
  The duplicated rows are not distinct neurons; how they enter the result is shown below.
- O1 (b), only the units above 0.5 Hz: the code's printed output gives 18 to 25 neurons per
  session, which the O1 (b) readings give and the O1 (a) readings do not (up to 34).
- O2 and O3 have no such basis. "Session-wide firing rate" (Methods, Secondary analysis) can be the
  rate over the recording, (a), but the same paragraph says the coupling was "estimated
  session-wide from activity recorded between the tactile stimulus onset and 2 s after the GO
  signal", which reads as (b). TSA is defined on CL, CR, IR and IL (Methods, Activity extraction),
  which supports O3 (a), but the claims file allows (b), and the code's output cannot tell them
  apart. Where an outcome depends on O2 or O3, the claim is blocked as `methods_underspecified`.

Each claim takes its own bootstrap generator, seeded from the claim, and is repeated under the four
further seeds (claim seed plus 1000 to 4000). The distribution of population coupling and the seven
bin edges are printed first.

In [ ]:
BIN_NAMES = ["0.00-14.29", "14.29-28.57", "28.57-42.86", "42.86-57.14", "57.14-71.43", "71.43-85.71", "85.71-100.00"]
PAPER_MD = [0.06, 0.004, 0.001, -0.002, 0.02, 0.06, 0.1]
PAPER_P = [7e-4, 0.12, 0.61, 0.76, 7e-4, 7e-4, 7e-4]

def run_claim(claim_id):
    seed = zlib.crc32(f"{PAPER}/{claim_id}".encode())
    out = {}
    for n in NAMES:
        for extra in [0, 1000, 2000, 3000, 4000]:
            rng = np.random.default_rng(seed + extra)
            out[(n, extra)] = ac.same_vs_diff_test(results[n]["same"], results[n]["diff"], rng)
    return seed, out

def table(out, extra=0):
    rows = []
    for n in NAMES:
        t = out[(n, extra)]
        for k in range(7):
            rows.append({"reading": n, "bin": BIN_NAMES[k], "n_same": t.n_same[k], "n_diff": t.n_diff[k],
                         "same": t.same_mean[k], "different": t.diff_mean[k],
                         "mean_difference": t.mean_difference[k], "p_holm": t.p_holm[k]})
    return pd.DataFrame(rows)

def seed_range(out, bins):
    rows = []
    for n in NAMES:
        for k in bins:
            md_ = [out[(n, e)].mean_difference[k] for e in [0, 1000, 2000, 3000, 4000]]
            p_ = [out[(n, e)].p_holm[k] for e in [0, 1000, 2000, 3000, 4000]]
            rows.append({"reading": n, "bin": BIN_NAMES[k], "md_min": min(md_), "md_max": max(md_),
                         "p_holm_min": min(p_), "p_holm_max": max(p_)})
    return pd.DataFrame(rows)

In [ ]:
for n in NAMES:
    c = np.concatenate(results[n]["coupling"])
    print(f"{n:6s} neurons {len(c)}, coupling median {np.median(c):.3f}, range {c.min():.3f} to {c.max():.3f}, "
          f"edges {np.round(results[n]['edges'], 3).tolist()}")
c = np.concatenate(results[MAIN]["coupling"])
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(c, bins=40, color="0.3")
for e in results[MAIN]["edges"]:
    ax.axvline(e, color="C1", lw=0.8)
ax.set_xlabel(f"population coupling (r), reading {MAIN}")
ax.set_ylabel("neurons")
plt.tight_layout()
plt.show()
print("all values are inside the axes; range", round(c.min(), 3), "to", round(c.max(), 3))

### Where the units stored twice fall

For each selected session of reading `baa-r` that holds a unit stored twice: the two rows' coupling
and bin, the correlation of their trial-specific activity, and the coupling and bin of the same unit
under `baa` (units counted once), where that session is selected too.

In [ ]:
by_path = {s["path"]: s for s in sessions}
once = results["baa"]
once_pos = {p: i for i, p in enumerate(once["sessions"].path)}
rows = []
for i, ss in enumerate(results[MAIN]["sessions"].itertuples()):
    ses = by_path[ss.path]
    pairs = ac.duplicate_pairs(ses, ss.units)
    if not pairs:
        continue
    x = ac.session_tsa(ses, ac.select_session(ses, *main, dedupe=False)).reshape(len(ss.units), -1)
    lab, cpl = results[MAIN]["labels"][i], results[MAIN]["coupling"][i]
    for a, b in pairs:
        row = {"session": f"{ss.subject} {ss.session}", "unit_id": int(ses["units"].id[ss.units[a]]),
               "coupling_rows": round(cpl[a], 3), "bin_rows": int(lab[a]) + 1, "twin_bin_rows": int(lab[b]) + 1,
               "r_twins": round(float(np.corrcoef(x[a], x[b])[0, 1]), 6)}
        if ss.path in once_pos:
            j = once_pos[ss.path]
            k = int(np.where(once["sessions"].units.iloc[j] == ss.units[a])[0][0])
            row.update({"coupling_once": round(once["coupling"][j][k], 3), "bin_once": int(once["labels"][j][k]) + 1})
        rows.append(row)
dups = pd.DataFrame(rows)
print(dups.to_string(index=False))
same_n = [sum(len(v) for v in results[MAIN]["same"][k]) for k in range(7)]
dup_bins = dups.bin_rows.value_counts().reindex(range(1, 8), fill_value=0)
print("\nduplicate pairs (r = 1) among the same-range pairs of each bin, reading", MAIN)
print(pd.DataFrame({"bin": BIN_NAMES, "duplicate_pairs": dup_bins.to_numpy(), "same_range_pairs": same_n}).to_string(index=False))

Each unit stored twice forms a pair with r = 1 with its copy, and both rows always fall in the same
bin. Its coupling rises when the copy is kept, as the copy is part of the summed activity of the
other neurons: under `baa-r` 12 of the 14 duplicated units are in the top bin and one each in bins 5
and 6. The 12 of them in sessions also selected under `baa` have a coupling of -0.089 to 0.128 when
counted once and fall in bins 1, 2, 4, 6 and 7. These r = 1 pairs are 12 of the 124 same-range
pairs of the top bin.

## figs4-lowest-couplers-correlated

> In the ALM silicon-probe recordings, neurons in the lowest population-coupling bin (0 to 14.29%)
> are more correlated with each other than with neurons of other coupling bins, as the synapses with
> the lowest coupling are.

Criterion: reproduced when the bin 1 mean difference is positive with Holm-corrected one-tailed
p < 0.05 and within 0.06 ± 0.006; partially reproduced when positive and significant but outside
tolerance; not reproduced when the Holm-corrected p is 0.05 or more.

In [ ]:
seed_low, out_low = run_claim("figs4-lowest-couplers-correlated")
t = table(out_low)
low = t[t.bin == BIN_NAMES[0]].copy()
low["outcome"] = np.where(low.p_holm >= 0.05, "not reproduced",
                          np.where((low.mean_difference > 0) & (abs(low.mean_difference - 0.06) <= 0.006), "reproduced",
                                   np.where(low.mean_difference > 0, "partially reproduced", "not reproduced")))
print(f"claim seed {seed_low}; paper: mean difference 0.06, Holm p 7.0e-4; the code printed 0.0575")
print(low.round(4).to_string(index=False))
print("\nover the five seeds:")
print(seed_range(out_low, [0]).round(4).to_string(index=False))
for n in [MAIN, "baa"]:
    print(f"\nall seven bins, reading {n}, claim seed (code_md: the mean difference printed by the paper's code):")
    print(out_low[(n, 0)].assign(bin=BIN_NAMES, paper_md=PAPER_MD, code_md=code_md, paper_p=PAPER_P,
                                 code_n_same=code_same).round(4).to_string(index=False))

In [ ]:
bm = ac.by_mouse_differences(results[MAIN])
print(f"reading {MAIN}: per mouse, mean same minus mean different correlation per bin (no test; n_same is the same-range pair count)")
print(bm.round(3).to_string(index=False))
print("\nmice with a positive bin 1 difference:", int((bm.bin1 > 0).sum()), "of", int(bm.bin1.notna().sum()))
fig, ax = plt.subplots(figsize=(6, 3))
tt = out_low[(MAIN, 0)]
ax.plot(range(1, 8), tt.same_mean, "ko", label="same range")
ax.plot(range(1, 8), tt.diff_mean, "^", color="0.5", label="different range")
ax.set_xticks(range(1, 8))
ax.set_xlabel("coupling bin (1 = lowest)")
ax.set_ylabel("mean pair correlation")
ax.legend()
plt.tight_layout()
plt.show()
print("plotted values range", round(float(min(tt.same_mean.min(), tt.diff_mean.min())), 4), "to",
      round(float(max(tt.same_mean.max(), tt.diff_mean.max())), 4), "; all inside the axes")

**Result: partially reproduced, under the assumed reading `baa-r`.** The lowest bin's mean
difference is 0.041 (same 0.022, different -0.019; Holm p = 0.0007), positive and significant but
below the tolerance of 0.054 to 0.066; over the five seeds it is 0.0402 to 0.0405. It is significant
under all sixteen readings (0.032 to 0.055) and within tolerance only under `bba` (0.0548), a reading
with units counted once; every reading with every stored row gives partially reproduced. The
difference is positive in each of the 8 mice with same-range pairs in the bin.

## figs4-high-couplers-correlated

> In the ALM silicon-probe recordings, neurons in the three highest coupling bins (57.14 to 100%)
> are more correlated with neurons of the same bin than with neurons of other bins (Table 5
> differences 0.02, 0.06 and 0.1), as for the synapses.

Criterion: reproduced when all three bins have a positive mean difference with Holm-corrected
one-tailed p < 0.05 and within tolerance (0.015 to 0.025, 0.054 to 0.066, 0.05 to 0.15);
partially reproduced when all three are significant and one or more is outside tolerance, or two of
three are significant and within tolerance.

In [ ]:
seed_high, out_high = run_claim("figs4-high-couplers-correlated")
t = table(out_high)
lo_hi = {4: (0.015, 0.025), 5: (0.054, 0.066), 6: (0.05, 0.15)}
rows = []
for n in NAMES:
    tt = out_high[(n, 0)]
    sig = [(tt.mean_difference[k] > 0) and (tt.p_holm[k] < 0.05) for k in (4, 5, 6)]
    tol = [lo_hi[k][0] <= tt.mean_difference[k] <= lo_hi[k][1] for k in (4, 5, 6)]
    if all(sig) and all(tol):
        o = "reproduced"
    elif all(sig) or sum(s and w for s, w in zip(sig, tol)) >= 2:
        o = "partially reproduced"
    else:
        o = "not reproduced"
    rows.append({"reading": n, **{f"md_{BIN_NAMES[k]}": tt.mean_difference[k] for k in (4, 5, 6)},
                 **{f"p_{BIN_NAMES[k]}": tt.p_holm[k] for k in (4, 5, 6)},
                 "significant": sum(sig), "within_tolerance": sum(tol), "outcome": o})
high = pd.DataFrame(rows)
print(f"claim seed {seed_high}; paper: 0.02, 0.06, 0.1, each Holm p 7.0e-4; the code printed {code_md[4:]}")
print(high.round(4).to_string(index=False))
print("\noutcomes by unit treatment:", high.groupby(high.reading.str.endswith("-r").map({True: "every stored row", False: "counted once"})).outcome.value_counts().to_dict())
print("\nover the five seeds:")
print(seed_range(out_high, [4, 5, 6]).round(4).to_string(index=False))
print(f"\npair counts, readings baa and {MAIN}:")
print(t[t.reading.isin(["baa", MAIN])][["reading", "bin", "n_same", "n_diff"]].to_string(index=False))

In [ ]:
print(f"reading {MAIN}, per mouse, bins 5 to 7 (no test):")
print(bm[["subject", "sessions", "bin5", "bin6", "bin7", "n_same5", "n_same6", "n_same7"]].round(3).to_string(index=False))
for k in (5, 6, 7):
    print(f"bin {k}: mice with a positive difference {int((bm[f'bin{k}'] > 0).sum())} of {int(bm[f'bin{k}'].notna().sum())}")

**Result: partially reproduced, under the assumed reading `baa-r`.** All three bins are significant
(mean differences 0.026, 0.035 and 0.130; Holm p 0.0007 for each, 0.0007 to 0.0020 over the five
seeds for the 57.14 to 71.43% bin), and only the top bin is within tolerance: 0.0255 is just above
the 0.015 to 0.025 interval and 0.035 is about half of the paper's 0.06. The four O1 (b) readings
with every stored row all give partially reproduced, as do `aaa-r`, `aab-r` and `aba-r`; `abb-r`
does not (57.14 to 71.43% bin Holm p 0.074). With units counted once, seven readings give not
reproduced, as the 57.14 to 71.43% bin is not significant, and `bab` gives partially reproduced. So
the outcome rests on keeping the duplicated rows, which put 12 pairs with r = 1 among the 124
same-range pairs of the top bin (above). The 71.43 to 85.71% difference is below the paper's 0.06
under every reading (0.014 to 0.035). The difference is positive in 7 of 9, 9 of 9 and 7 of 7 mice
in the three bins under `baa-r`; in two mice the only top-bin same-range pair is a unit with its
copy (0.991 and 0.997).

## figs4-middle-couplers-not-correlated

> In the ALM silicon-probe recordings, the three middle coupling bins (14.29 to 57.14%) show no
> significant excess of same-bin over different-bin correlation.

Criterion: a claim of absence on the paper's test (E8, one-tailed, Holm across the 7 bins,
alpha 0.05). Reproduced when none of bins 2, 3 and 4 has a Holm-corrected one-tailed p below
0.05; not reproduced when any does. The three mean differences are reported either way.

In [ ]:
seed_mid, out_mid = run_claim("figs4-middle-couplers-not-correlated")
rows = []
for n in NAMES:
    tt = out_mid[(n, 0)]
    rows.append({"reading": n, **{f"md_{BIN_NAMES[k]}": tt.mean_difference[k] for k in (1, 2, 3)},
                 **{f"p_{BIN_NAMES[k]}": tt.p_holm[k] for k in (1, 2, 3)},
                 "outcome": "reproduced" if all(tt.p_holm[k] >= 0.05 for k in (1, 2, 3)) else "not reproduced"})
mid = pd.DataFrame(rows)
print(f"claim seed {seed_mid}; paper: 0.004 (p 0.12), 0.001 (p 0.61), -0.002 (p 0.76); the code printed {code_md[1:4]}")
print(mid.round(4).to_string(index=False))
print("\noutcome by reading:", mid.set_index("reading").outcome.to_dict())
print("\nover the five seeds:")
print(seed_range(out_mid, [1, 2, 3]).round(4).to_string(index=False))
print(f"\nreading {MAIN}, per mouse (no test):")
print(bm[["subject", "bin2", "bin3", "bin4"]].round(3).to_string(index=False))

**Result: blocked (`methods_underspecified`).** The outcome depends on O2, for which the paper gives
no defensible choice. With every stored row and O1 (b), `baa-r` and `bab-r` give reproduced (smallest
Holm p 0.064, for 14.29 to 28.57% under `baa-r`, 0.061 to 0.068 over the five seeds), while `bba-r`
and `bbb-r` give not reproduced (14.29 to 28.57%: 0.008 with Holm p 0.0054, and 0.007 with Holm p
0.0008). Over all sixteen readings eight give reproduced and eight not reproduced. Where a middle bin
is significant, the mean difference is at most 0.0084; the paper's are 0.004, 0.001 and -0.002.

In [ ]:
print(f"notebook run time {(time.time() - started) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Summary

| Claim | Outcome |
|---|---|
| figs4-session-selection | not reproduced (9 or 10 mice and 15 to 18 sessions, against 11 and 20) |
| figs4-lowest-couplers-correlated | partially reproduced, assumed reading `baa-r` (0.041 against 0.06, significant) |
| figs4-high-couplers-correlated | partially reproduced, assumed reading `baa-r` (all three significant; 0.026, 0.035 and 0.130 against 0.02, 0.06 and 0.1) |
| figs4-middle-couplers-not-correlated | blocked (`methods_underspecified`): reproduced under eight readings and not under eight; it turns on O2 |
| 30 claims on the paper's imaging | blocked (`data_missing`), the data are not public |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The step that builds the paper's per-session arrays from
these files is not in its code, so the unit filter (O1), the rate (O2), the trial set (O3) and the
treatment of units stored twice are run under every combination. Where outcomes differ, every stored
row is kept and only units above 0.5 Hz enter (reading `baa-r`), as assumptions with a stated basis;
the window is a fixed 4.6 s from sample onset (O5). The selection does not reach the paper's 11 mice
and 20 sessions under any reading, so the population differs from the paper's, which bears on the
three test claims.

What was found about the dataset. Every unit is labelled "good"; 50 of 99 files store 72 units
twice under a repeated id, and an analysis that keeps both rows puts these units, paired with their
copies at r = 1, mostly in the top coupling bin; every trial is labelled `audio delay` though the
task described is tactile; some event timestamps are NaN; a few trials with sample times have no
observation interval; a few units store spike times out of order, and two files hold spikes outside
the observation intervals. The spikes and trial events share a clock as far as the decoding check
shows, and photostimulation onsets agree with the stored laser trace in the selected sessions that
have one; the selected sessions of Ai32 mice hold no laser trace or onsets.